In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})  # TrueType text in PDFs (editable in Illustrator/Inkscape)
import seaborn as sns

from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
# import pingouin as pg

import sys
sys.path.append('/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/')
from clinical_data_analysis.statistical_analysis.settings import *

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
SAVE_DIR = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/figures/'

In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name='1')

df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]

df.rename(columns={'#':'ID',
                    'תאריך בדיקות דם': 'blood_test_date',
                    'שעת לקיחת דם':'blood_taken_time',
                    'מספר מנות':'num_of_injections',
                    'גיל הפסקת מחזור':'menopause',
                    'אסטרוגן':'estorgen',
                    'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                    'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                    'p-tau_ new_#1':'p-tau_new_#1',
                    'ביקורת אחרי שנה':'checkup after a year',
                    'חיסון':'vaccine',
                    'תרופות':'medications',
                    'הערות':'notes',
                    'Number_of_risk_faktors':'Number_of_risk_factors',
                    'P-tau217':'p-tau217',
                    'צרבונין':'Cerebonin',
                    'Spatial ': 'Spatial'
                    }, inplace=True)
df = df.apply(pd.to_numeric, errors='ignore')
df = df.drop(df[df['ID'].isin(OUTLIERS)].index)

In [ ]:
df = df.drop(df[df['ID'].isin(OUTLIERS)].index)
df = df[df[MOCA_COL] != '?']

In [ ]:
negative_columns = ['Negative MOCA_score', 'Negative Executive', 'Negative Spatial', 'Negative Attention_and_concentration', 'Negative Memory', 'Negative Language', 'Negative Orientation', 'Negative IST']

df[negative_columns] = df[['MOCA_score', 'Executive', 'Spatial', 'Attention_and_concentration', 'Memory', 'Language', 'Orientation', 'IST']].apply(lambda x: x.astype(float) * -1)
df[negative_columns].head(1)

In [ ]:
cognitive_tests = negative_columns + ['Frontal memory score (#1)', 'Hipocampal memory score (#2)', 'Trail_B_(second)']
behavioral_tests = ['Mild_behavioral_Impairment_Checklist', 'interest, motivation, and drive', 'mood or anxiety symptoms', 'Impulse control', 'following societal norms', 'strongly held beliefs and sensory', 'iADL']

risk_columns = ['height', 'Weight', 'BMI', 'Heart_disease', 'Diabetes', 'Hypertension', 'Hyperlipidemia',
                'Periodontal disease', 'Recurrent_UTI/_pneumonia/other_infection', 'Systemic_inflammatory_disease', 'Smoking', 'Pack_years',
                'Alcohol_consumption', 'Insomnia', 'OSA', 'Trearment_for_OSA', 'severity of OSA', 'Physical activity', 'Time_Physical_activity_(hours/week)',
                'Anxiety/depression','Cognitive_activity', 'AD_in_the_family', 'Number_of_risk_factors']

In [ ]:
TAU_COLUMNS = [TAU_217]
COGNITIVE_COLUMN = 'Trail_B_(second)'
BEHAVIORAL_COLUMN = 'Mild_behavioral_Impairment_Checklist'
# COLUMNS_TO_CORRECT = ['Gender', 'Age']
print(f"Before: {df.shape}")
df = df.dropna(subset=[COGNITIVE_COLUMN, BEHAVIORAL_COLUMN, TAU_217])
print(f"After: {df.shape}")
# remove the samples with MOCA score below 18
df = df[df[MOCA_COL] >= 19]
print(f"After removing samples with MOCA score below 18: {df.shape}")


In [ ]:
df[COGNITIVE_COLUMN] = df[COGNITIVE_COLUMN].astype(int)
df[MCI] = (df[COGNITIVE_COLUMN] < MCI_THR).astype(int)

df[BEHAVIORAL_COLUMN] = df[BEHAVIORAL_COLUMN].astype(int)

In [ ]:
c_b_df = df[cognitive_tests + behavioral_tests + [TAU_217]]# + COLUMNS_TO_CORRECT]
# relevant_cols = [col for col in c_b_df.columns if col not in COLUMNS_TO_CORRECT]
# for col in relevant_cols:
#     c_b_df[col] = pd.to_numeric(c_b_df[col], errors='coerce')

# print(c_b_df.shape)
# c_b_df = c_b_df[c_b_df[['Negative Memory', 'Trail_B_(second)']].notna().all(axis=1)]
# print(c_b_df.shape)


In [ ]:
c_b_df['Negative MOCA_score'].max()

# Noramlize the c_b_df dataset

In [ ]:
# # # Standardize to help the k-NN MI estimator
# X = c_b_df.values.astype(float)
# X = StandardScaler().fit_transform(X)
# c_b_df = pd.DataFrame(X, columns=c_b_df.columns)

# Correlation analysis

In [ ]:
c_b_df

In [ ]:
# Scale the c_b_df and high_p_tau_c_b_df and the healhty_c_b_df
c_b_df_scaled = c_b_df.copy()
high_p_tau_c_b_df_scaled = high_p_tau_c_b_df.copy()
c_b_df_scaled[cognitive_tests + behavioral_tests] = StandardScaler().fit_transform(c_b_df[cognitive_tests + behavioral_tests])
high_p_tau_c_b_df_scaled[cognitive_tests + behavioral_tests] = StandardScaler().fit_transform(high_p_tau_c_b_df[cognitive_tests + behavioral_tests])


In [ ]:
FEATURE_ORDER = [
    'Negative MOCA_score',
    'Negative Executive',
    'Negative Spatial',
    'Negative Attention_and_concentration',
    'Negative Language',
    'Negative Orientation',
    'Negative Memory',
    'Frontal memory score (#1)',
    'Hipocampal memory score (#2)',
    'Negative IST',
    'Trail_B_(second)',
    'Mild_behavioral_Impairment_Checklist',
    'interest, motivation, and drive',
    'mood or anxiety symptoms',
    'Impulse control',
    'following societal norms',
    'strongly held beliefs and sensory',
    'iADL',
]

FEATURE_LABELS = {
    'Negative MOCA_score': 'MoCA',
    'Negative Executive': 'Executive function',
    'Negative Spatial': 'Visuospatial',
    'Negative Attention_and_concentration': 'Attention & concentration',
    'Negative Language': 'Language',
    'Negative Orientation': 'Orientation',
    'Negative Memory': 'Memory',
    'Frontal memory score (#1)': 'Frontal memory',
    'Hipocampal memory score (#2)': 'Hippocampal memory',
    'Negative IST': 'Word Fluency',
    'Trail_B_(second)': 'Trail B',
    'Mild_behavioral_Impairment_Checklist': 'MBI-C',
    'interest, motivation, and drive': 'Motivation',
    'mood or anxiety symptoms': 'Mood',
    'Impulse control': 'Impulsivity',
    'following societal norms': 'Conformity',
    'strongly held beliefs and sensory': 'Held beliefs',
    'iADL': 'iADL',
}

c_b_df_plot = c_b_df[FEATURE_ORDER].rename(columns=FEATURE_LABELS)

cols = c_b_df_plot.columns
corr_matrix = c_b_df_plot.corr()
pval_matrix = pd.DataFrame(np.ones(corr_matrix.shape), columns=cols, index=cols)

for i in range(len(cols)):
    for j in range(len(cols)):
        if i != j:
            r, p = pearsonr(c_b_df_plot[cols[i]].astype(float), c_b_df_plot[cols[j]].astype(float))
            pval_matrix.iloc[i, j] = p
        # else:
        #     pval_matrix.iloc[i, j] = np.nan

annot_matrix = corr_matrix.round(2).astype(str)
for i in range(len(cols)):
    for j in range(len(cols)):
        p = pval_matrix.iloc[i, j]
        if p < 0.005:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n***"
        elif p < 0.01:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n**"
        elif p < 0.05:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n*"
        else:
            annot = f"{corr_matrix.iloc[i, j]:.2f}"
        annot_matrix.iloc[i, j] = annot

In [ ]:
plt.figure(figsize=(15, 15))

sns.heatmap(corr_matrix, annot=annot_matrix, vmin=-1, vmax=1, cmap='coolwarm', fmt='', rasterized=False)
# plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests N={c_b_df.shape[0]}')
plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests\n N={len(c_b_df)}', fontsize=18)

plt.savefig(f'{SAVE_DIR}/Cognitive_and_Behavioral_pearson_correlation_heatmap.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
FEATURE_ORDER = [
    'Negative MOCA_score',
    'Negative Executive',
    'Negative Spatial',
    'Negative Attention_and_concentration',
    'Negative Language',
    'Negative Orientation',
    'Negative Memory',
    'Frontal memory score (#1)',
    'Hipocampal memory score (#2)',
    'Negative IST',
    'Trail_B_(second)',
    'Mild_behavioral_Impairment_Checklist',
    'interest, motivation, and drive',
    'mood or anxiety symptoms',
    'Impulse control',
    'following societal norms',
    'strongly held beliefs and sensory',
    'iADL',
]

FEATURE_LABELS = {
    'Negative MOCA_score': 'MoCA',
    'Negative Executive': 'Executive function',
    'Negative Spatial': 'Visuospatial',
    'Negative Attention_and_concentration': 'Attention & concentration',
    'Negative Language': 'Language',
    'Negative Orientation': 'Orientation',
    'Negative Memory': 'Memory',
    'Frontal memory score (#1)': 'Frontal memory',
    'Hipocampal memory score (#2)': 'Hippocampal memory',
    'Negative IST': 'Word Fluency',
    'Trail_B_(second)': 'Trail B',
    'Mild_behavioral_Impairment_Checklist': 'MBI-C',
    'interest, motivation, and drive': 'Motivation',
    'mood or anxiety symptoms': 'Mood',
    'Impulse control': 'Impulsivity',
    'following societal norms': 'Conformity',
    'strongly held beliefs and sensory': 'Held beliefs',
    'iADL': 'iADL',
}

c_b_df_plot = c_b_df[FEATURE_ORDER].rename(columns=FEATURE_LABELS)

cols = c_b_df_plot.columns
corr_matrix = c_b_df_plot.corr()
pval_matrix = pd.DataFrame(np.ones(corr_matrix.shape), columns=cols, index=cols)

for i in range(len(cols)):
    for j in range(len(cols)):
        if i != j:
            r, p = pearsonr(c_b_df_plot[cols[i]].astype(float), c_b_df_plot[cols[j]].astype(float))
            pval_matrix.iloc[i, j] = p
        # else:
        #     pval_matrix.iloc[i, j] = np.nan

annot_matrix = corr_matrix.round(2).astype(str)
for i in range(len(cols)):
    for j in range(len(cols)):
        p = pval_matrix.iloc[i, j]
        if p < 0.005:
            annot = f"***"
        elif p < 0.01:
            annot = f"**"
        elif p < 0.05:
            annot = f"*"
        else:
            annot = f""
        annot_matrix.iloc[i, j] = annot


plt.figure(figsize=(15, 15))


sns.heatmap(corr_matrix, annot=annot_matrix, vmin=-1, vmax=1, cmap='coolwarm', fmt='', rasterized=False)
# plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests N={c_b_df.shape[0]}')
plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests\n N={len(c_b_df)}', fontsize=18)

plt.savefig(f'{SAVE_DIR}/Cognitive_and_Behavioral_pearson_correlation_heatmap_starts_only.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
FEATURE_ORDER = [
    'Negative MOCA_score',
    'Negative Executive',
    'Negative Spatial',
    'Negative Attention_and_concentration',
    'Negative Language',
    'Negative Orientation',
    'Negative Memory',
    'Frontal memory score (#1)',
    'Hipocampal memory score (#2)',
    'Negative IST',
    'Trail_B_(second)',
    'Mild_behavioral_Impairment_Checklist',
    'interest, motivation, and drive',
    'mood or anxiety symptoms',
    'Impulse control',
    'following societal norms',
    'strongly held beliefs and sensory',
    'iADL',
]

FEATURE_LABELS = {
    'Negative MOCA_score': 'MoCA',
    'Negative Executive': 'Executive function',
    'Negative Spatial': 'Visuospatial',
    'Negative Attention_and_concentration': 'Attention & concentration',
    'Negative Language': 'Language',
    'Negative Orientation': 'Orientation',
    'Negative Memory': 'Memory',
    'Frontal memory score (#1)': 'Frontal memory',
    'Hipocampal memory score (#2)': 'Hippocampal memory',
    'Negative IST': 'Word Fluency',
    'Trail_B_(second)': 'Trail B',
    'Mild_behavioral_Impairment_Checklist': 'MBI-C',
    'interest, motivation, and drive': 'Motivation',
    'mood or anxiety symptoms': 'Mood',
    'Impulse control': 'Impulsivity',
    'following societal norms': 'Conformity',
    'strongly held beliefs and sensory': 'Held beliefs',
    'iADL': 'iADL',
}

c_b_df_plot = c_b_df[FEATURE_ORDER].rename(columns=FEATURE_LABELS)

cols = c_b_df_plot.columns
corr_matrix = c_b_df_plot.corr()
pval_matrix = pd.DataFrame(np.ones(corr_matrix.shape), columns=cols, index=cols)

for i in range(len(cols)):
    for j in range(len(cols)):
        if i != j:
            r, p = pearsonr(c_b_df_plot[cols[i]].astype(float), c_b_df_plot[cols[j]].astype(float))
            pval_matrix.iloc[i, j] = p
        # else:
        #     pval_matrix.iloc[i, j] = np.nan

annot_matrix = corr_matrix.round(2).astype(str)


plt.figure(figsize=(15, 15))


sns.heatmap(corr_matrix, annot=annot_matrix, vmin=-1, vmax=1, cmap='coolwarm', fmt='', rasterized=False)
# plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests N={c_b_df.shape[0]}')
plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests\n N={len(c_b_df)}', fontsize=18)

plt.savefig(f'{SAVE_DIR}/Cognitive_and_Behavioral_pearson_correlation_heatmap_numbers_only.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
c_b_df.columns

In [ ]:
#scale c_b_df_healthy
c_b_df_healthy_scaled = c_b_df_healthy.copy()
c_b_df_healthy_scaled[cognitive_tests + behavioral_tests] = StandardScaler().fit_transform(c_b_df_healthy[cognitive_tests + behavioral_tests])


In [ ]:
from matplotlib import colors
import numpy as np
from scipy.stats import linregress

# Columns to compare
cognitive_column = TMT_B_COL
behavioral_column = MBI_C_COL

# Filter to remove max values in cognitive column from both dfs, make copies to avoid affecting original data
healthy_scatter_df = c_b_df_healthy[c_b_df_healthy[behavioral_column] != c_b_df_healthy[behavioral_column].max()].dropna(subset=[cognitive_column, behavioral_column])
high_p_tau_scatter_df = c_b_df_high_p_tau[c_b_df_high_p_tau[behavioral_column] != c_b_df_high_p_tau[behavioral_column].max()].dropna(subset=[cognitive_column, behavioral_column])

# Calculate correlation for each
correlation_healthy = healthy_scatter_df[cognitive_column].corr(healthy_scatter_df[behavioral_column])
correlation_high = high_p_tau_scatter_df[cognitive_column].corr(high_p_tau_scatter_df[behavioral_column])

print(f"Correlation (healthy) between {cognitive_column} and {behavioral_column}: {correlation_healthy}")
print(f"Correlation (high_p_tau) between {cognitive_column} and {behavioral_column}: {correlation_high}")

# Setup figure and axes for two plots side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
norm = colors.Normalize(vmin=0, vmax=1, clip=True)

# ========== Healthy ==============
ax = axes[0]
sc = ax.scatter(
    healthy_scatter_df[cognitive_column],
    healthy_scatter_df[behavioral_column],
    alpha=0.2,
    norm=norm,
)
x1 = healthy_scatter_df[cognitive_column]
y1 = healthy_scatter_df[behavioral_column]
slope1, intercept1, r_value1, p_value1, std_err1 = linregress(x1, y1)
x1_line = np.linspace(x1.min(), x1.max(), 100)
y1_line = slope1 * x1_line + intercept1
ax.plot(x1_line, y1_line, color='black', linestyle='--', label=f'Corr line (r={correlation_healthy:.2f})')
ax.set_xlabel(cognitive_column)
ax.set_ylabel(behavioral_column)
ax.set_title(f'Healthy group\nN={healthy_scatter_df.shape[0]}, Corr={correlation_healthy:.2f}')
ax.legend()

# ========== High p-tau ==============
ax = axes[1]
sc2 = ax.scatter(
    high_p_tau_scatter_df[cognitive_column],
    high_p_tau_scatter_df[behavioral_column],
    alpha=0.2,
    norm=norm,
)
x2 = high_p_tau_scatter_df[cognitive_column]
y2 = high_p_tau_scatter_df[behavioral_column]
slope2, intercept2, r_value2, p_value2, std_err2 = linregress(x2, y2)
x2_line = np.linspace(x2.min(), x2.max(), 100)
y2_line = slope2 * x2_line + intercept2
ax.plot(x2_line, y2_line, color='black', linestyle='--', label=f'Corr line (r={correlation_high:.2f})')
ax.set_xlabel(cognitive_column)
ax.set_ylabel(behavioral_column)
ax.set_title(f'High p-tau217 group\nN={high_p_tau_scatter_df.shape[0]}, Corr={correlation_high:.2f}')
ax.legend()

plt.suptitle(f'Scatter plot of {cognitive_column} score vs {behavioral_column}\n(Left: Healthy, Right: High p-tau217)')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


### Same but only for cognitive healthy. MCI=0

In [ ]:
# c_b_df_healthy = c_b_df[c_b_df["Negative MOCA_score"] <= -1 * MCI_THR]
c_b_df = df[cognitive_tests + behavioral_tests  + [TAU_217]]
c_b_df = c_b_df[c_b_df[TAU_217].notna()]
c_b_df_high_p_tau = c_b_df[c_b_df[TAU_217] > HIGH_TAU_217_THR]
c_b_df_healthy = c_b_df[c_b_df[TAU_217] <= HIGH_TAU_217_THR]
c_b_df_healthy = c_b_df_healthy[c_b_df_healthy["Negative MOCA_score"] <= -1 * MCI_THR]
# remove tau_217 from c_b_df_healthy and c_b_df_high_p_tau
c_b_df_healthy = c_b_df_healthy.drop(columns=[TAU_217])
c_b_df_high_p_tau = c_b_df_high_p_tau.drop(columns=[TAU_217])
# remove rows with nan values in iADL, memory or trail_b from c_b_df_high_p_tau and c_b_df_healthy
c_b_df_high_p_tau = c_b_df_high_p_tau[c_b_df_high_p_tau["iADL"].notna()]
c_b_df_high_p_tau = c_b_df_high_p_tau[c_b_df_high_p_tau["Negative Memory"].notna()]
c_b_df_high_p_tau = c_b_df_high_p_tau[c_b_df_high_p_tau["Trail_B_(second)"].notna()]
c_b_df_healthy = c_b_df_healthy[c_b_df_healthy["iADL"].notna()]
c_b_df_healthy = c_b_df_healthy[c_b_df_healthy["Negative Memory"].notna()]
c_b_df_healthy = c_b_df_healthy[c_b_df_healthy["Trail_B_(second)"].notna()]

In [ ]:
cols = c_b_df_healthy.columns
corr_matrix = c_b_df_healthy.corr()
pval_matrix = pd.DataFrame(np.ones(corr_matrix.shape), columns=cols, index=cols)

for i in range(len(cols)):
    for j in range(len(cols)):
        if i != j:
            r, p = pearsonr(c_b_df_healthy[cols[i]].astype(float), c_b_df_healthy[cols[j]].astype(float))
            pval_matrix.iloc[i, j] = p
        else:
            pval_matrix.iloc[i, j] = np.nan

annot_matrix = corr_matrix.round(2).astype(str)
for i in range(len(cols)):
    for j in range(len(cols)):
        p = pval_matrix.iloc[i, j]

        if np.isnan(p):
            annot = ""
            # annot = f"{corr_matrix.iloc[i, j]:.2f}\nNA"
        elif p < 0.005:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n***"
        elif p < 0.01:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n**"
        elif p < 0.05:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n*"
        else:
            annot = f"{corr_matrix.iloc[i, j]:.2f}"
        annot_matrix.iloc[i, j] = annot

In [ ]:
plt.figure(figsize=(15, 15))

sns.heatmap(c_b_df_healthy.corr(), annot=annot_matrix, vmin=-1, vmax=1, cmap='coolwarm', fmt='', rasterized=False)
# plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests N={c_b_df.shape[0]}')
plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests\n N={len(c_b_df_healthy)} - Cognitive healthy', fontsize=18)
plt.savefig(f'{SAVE_DIR}/Cognitive_and_Behavioral_heatmap_pearson_correlation_heatmap_cognitive_healthy.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
c_b_df_mci = c_b_df[c_b_df["Negative MOCA_score"] > -1 * MCI_THR]
cols = c_b_df_mci.columns
corr_matrix = c_b_df_mci.corr()
pval_matrix = pd.DataFrame(np.ones(corr_matrix.shape), columns=cols, index=cols)

for i in range(len(cols)):
    for j in range(len(cols)):
        if i != j:
            r, p = pearsonr(c_b_df_mci[cols[i]].astype(float), c_b_df_mci[cols[j]].astype(float))
            pval_matrix.iloc[i, j] = p
        else:
            pval_matrix.iloc[i, j] = np.nan

annot_matrix = corr_matrix.round(2).astype(str)
for i in range(len(cols)):
    for j in range(len(cols)):
        p = pval_matrix.iloc[i, j]

        if np.isnan(p):
            annot = ""
            # annot = f"{corr_matrix.iloc[i, j]:.2f}\nNA"
        elif p < 0.005:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n***"
        elif p < 0.01:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n**"
        elif p < 0.05:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n*"
        else:
            annot = f"{corr_matrix.iloc[i, j]:.2f}"
        annot_matrix.iloc[i, j] = annot

plt.figure(figsize=(15, 15))

sns.heatmap(c_b_df_mci.corr(), annot=annot_matrix, vmin=-1, vmax=1, cmap='coolwarm', fmt='', rasterized=False)
# plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests N={c_b_df.shape[0]}')
plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests\n N={len(c_b_df_mci)} - MCI', fontsize=18)
plt.savefig(f'{SAVE_DIR}/Cognitive_and_Behavioral_heatmap_pearson_correlation_heatmap_MCI.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# Same but for high p-tau individuals

c_b_df_high_p_tau = c_b_df[c_b_df[TAU_217] > HIGH_TAU_217_THR]
c_b_df_high_p_tau = c_b_df_high_p_tau[c_b_df_high_p_tau[TAU_217].notna()]
c_b_df_high_p_tau = c_b_df_high_p_tau.drop(columns=[TAU_217])

FEATURE_ORDER = [
    'Negative MOCA_score',
    'Negative Executive',
    'Negative Spatial',
    'Negative Attention_and_concentration',
    'Negative Language',
    'Negative Orientation',
    'Negative Memory',
    'Frontal memory score (#1)',
    'Hipocampal memory score (#2)',
    'Negative IST',
    'Trail_B_(second)',
    'Mild_behavioral_Impairment_Checklist',
    'interest, motivation, and drive',
    'mood or anxiety symptoms',
    'Impulse control',
    'following societal norms',
    'strongly held beliefs and sensory',
    'iADL',
]

FEATURE_LABELS = {
    'Negative MOCA_score': 'MoCA',
    'Negative Executive': 'Executive function',
    'Negative Spatial': 'Visuospatial',
    'Negative Attention_and_concentration': 'Attention & concentration',
    'Negative Language': 'Language',
    'Negative Orientation': 'Orientation',
    'Negative Memory': 'Memory',
    'Frontal memory score (#1)': 'Frontal memory',
    'Hipocampal memory score (#2)': 'Hippocampal memory',
    'Negative IST': 'Word Fluency',
    'Trail_B_(second)': 'Trail B',
    'Mild_behavioral_Impairment_Checklist': 'MBI-C',
    'interest, motivation, and drive': 'Motivation',
    'mood or anxiety symptoms': 'Mood',
    'Impulse control': 'Impulsivity',
    'following societal norms': 'Conformity',
    'strongly held beliefs and sensory': 'Held beliefs',
    'iADL': 'iADL',
}

c_b_df_high_p_tau = c_b_df_high_p_tau[FEATURE_ORDER].rename(columns=FEATURE_LABELS)

cols = c_b_df_high_p_tau.columns
corr_matrix = c_b_df_high_p_tau.corr()
pval_matrix = pd.DataFrame(np.ones(corr_matrix.shape), columns=cols, index=cols)

for i in range(len(cols)):
    for j in range(len(cols)):
        if i != j:
            r, p = pearsonr(c_b_df_high_p_tau[cols[i]].astype(float), c_b_df_high_p_tau[cols[j]].astype(float))
            pval_matrix.iloc[i, j] = p
        else:
            pval_matrix.iloc[i, j] = np.nan

annot_matrix = corr_matrix.round(2).astype(str)
for i in range(len(cols)):
    for j in range(len(cols)):
        p = pval_matrix.iloc[i, j]

        if np.isnan(p):
            annot = ""
            # annot = f"{corr_matrix.iloc[i, j]:.2f}\nNA"
        elif p < 0.005:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n***"
        elif p < 0.01:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n**"
        elif p < 0.05:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n*"
        else:
            annot = f"{corr_matrix.iloc[i, j]:.2f}"
        annot_matrix.iloc[i, j] = annot

plt.figure(figsize=(15, 15))

sns.heatmap(c_b_df_high_p_tau.corr(), annot=annot_matrix, vmin=-1, vmax=1, cmap='coolwarm', fmt='', rasterized=False)
# plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests N={c_b_df.shape[0]}')
plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests\n N={len(c_b_df_high_p_tau)} - High p-tau', fontsize=18)
plt.savefig(f'{SAVE_DIR}/Cognitive_and_Behavioral_pearson_correlation_heatmap__high_p_tau.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# Same but for high p-tau individuals

c_b_df_high_p_tau = c_b_df[c_b_df[TAU_217] > HIGH_TAU_217_THR]
c_b_df_high_p_tau = c_b_df_high_p_tau[c_b_df_high_p_tau[TAU_217].notna()]
c_b_df_high_p_tau = c_b_df_high_p_tau.drop(columns=[TAU_217])

FEATURE_ORDER = [
    'Negative MOCA_score',
    'Negative Executive',
    'Negative Spatial',
    'Negative Attention_and_concentration',
    'Negative Language',
    'Negative Orientation',
    'Negative Memory',
    'Frontal memory score (#1)',
    'Hipocampal memory score (#2)',
    'Negative IST',
    'Trail_B_(second)',
    'Mild_behavioral_Impairment_Checklist',
    'interest, motivation, and drive',
    'mood or anxiety symptoms',
    'Impulse control',
    'following societal norms',
    'strongly held beliefs and sensory',
    'iADL',
]

FEATURE_LABELS = {
    'Negative MOCA_score': 'MoCA',
    'Negative Executive': 'Executive function',
    'Negative Spatial': 'Visuospatial',
    'Negative Attention_and_concentration': 'Attention & concentration',
    'Negative Language': 'Language',
    'Negative Orientation': 'Orientation',
    'Negative Memory': 'Memory',
    'Frontal memory score (#1)': 'Frontal memory',
    'Hipocampal memory score (#2)': 'Hippocampal memory',
    'Negative IST': 'Word Fluency',
    'Trail_B_(second)': 'Trail B',
    'Mild_behavioral_Impairment_Checklist': 'MBI-C',
    'interest, motivation, and drive': 'Motivation',
    'mood or anxiety symptoms': 'Mood',
    'Impulse control': 'Impulsivity',
    'following societal norms': 'Conformity',
    'strongly held beliefs and sensory': 'Held beliefs',
    'iADL': 'iADL',
}

c_b_df_high_p_tau = c_b_df_high_p_tau[FEATURE_ORDER].rename(columns=FEATURE_LABELS)

cols = c_b_df_high_p_tau.columns
corr_matrix = c_b_df_high_p_tau.corr()
pval_matrix = pd.DataFrame(np.ones(corr_matrix.shape), columns=cols, index=cols)

for i in range(len(cols)):
    for j in range(len(cols)):
        if i != j:
            r, p = pearsonr(c_b_df_high_p_tau[cols[i]].astype(float), c_b_df_high_p_tau[cols[j]].astype(float))
            pval_matrix.iloc[i, j] = p
        else:
            pval_matrix.iloc[i, j] = np.nan

annot_matrix = corr_matrix.round(2).astype(str)


plt.figure(figsize=(15, 15))

sns.heatmap(c_b_df_high_p_tau.corr(), annot=annot_matrix, vmin=-1, vmax=1, cmap='coolwarm', fmt='', rasterized=False)
# plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests N={c_b_df.shape[0]}')
plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests\n N={len(c_b_df_high_p_tau)} - High p-tau', fontsize=18)
plt.savefig(f'{SAVE_DIR}/Cognitive_and_Behavioral_pearson_correlation_heatmap__high_p_tau_numbers_only.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# Same but for high p-tau individuals

c_b_df_high_p_tau = c_b_df[c_b_df[TAU_217] > HIGH_TAU_217_THR]
c_b_df_high_p_tau = c_b_df_high_p_tau[c_b_df_high_p_tau[TAU_217].notna()]
c_b_df_high_p_tau = c_b_df_high_p_tau.drop(columns=[TAU_217])

FEATURE_ORDER = [
    'Negative MOCA_score',
    'Negative Executive',
    'Negative Spatial',
    'Negative Attention_and_concentration',
    'Negative Language',
    'Negative Orientation',
    'Negative Memory',
    'Frontal memory score (#1)',
    'Hipocampal memory score (#2)',
    'Negative IST',
    'Trail_B_(second)',
    'Mild_behavioral_Impairment_Checklist',
    'interest, motivation, and drive',
    'mood or anxiety symptoms',
    'Impulse control',
    'following societal norms',
    'strongly held beliefs and sensory',
    'iADL',
]

FEATURE_LABELS = {
    'Negative MOCA_score': 'MoCA',
    'Negative Executive': 'Executive function',
    'Negative Spatial': 'Visuospatial',
    'Negative Attention_and_concentration': 'Attention & concentration',
    'Negative Language': 'Language',
    'Negative Orientation': 'Orientation',
    'Negative Memory': 'Memory',
    'Frontal memory score (#1)': 'Frontal memory',
    'Hipocampal memory score (#2)': 'Hippocampal memory',
    'Negative IST': 'Word Fluency',
    'Trail_B_(second)': 'Trail B',
    'Mild_behavioral_Impairment_Checklist': 'MBI-C',
    'interest, motivation, and drive': 'Motivation',
    'mood or anxiety symptoms': 'Mood',
    'Impulse control': 'Impulsivity',
    'following societal norms': 'Conformity',
    'strongly held beliefs and sensory': 'Held beliefs',
    'iADL': 'iADL',
}

c_b_df_high_p_tau = c_b_df_high_p_tau[FEATURE_ORDER].rename(columns=FEATURE_LABELS)

cols = c_b_df_high_p_tau.columns
corr_matrix = c_b_df_high_p_tau.corr()
pval_matrix = pd.DataFrame(np.ones(corr_matrix.shape), columns=cols, index=cols)

for i in range(len(cols)):
    for j in range(len(cols)):
        if i != j:
            r, p = pearsonr(c_b_df_high_p_tau[cols[i]].astype(float), c_b_df_high_p_tau[cols[j]].astype(float))
            pval_matrix.iloc[i, j] = p
        else:
            pval_matrix.iloc[i, j] = np.nan

annot_matrix = corr_matrix.round(2).astype(str)
for i in range(len(cols)):
    for j in range(len(cols)):
        p = pval_matrix.iloc[i, j]

        if np.isnan(p):
            annot = ""
            # annot = f"{corr_matrix.iloc[i, j]:.2f}\nNA"
        elif p < 0.005:
            annot = f"***"
        elif p < 0.01:
            annot = f"**"
        elif p < 0.05:
            annot = f"*"
        else:
            annot = f""
        annot_matrix.iloc[i, j] = annot

plt.figure(figsize=(15, 15))

sns.heatmap(c_b_df_high_p_tau.corr(), annot=annot_matrix, vmin=-1, vmax=1, cmap='coolwarm', fmt='', rasterized=False)
# plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests N={c_b_df.shape[0]}')
plt.title(f'Pearson correlation heatmap of cognitive and behavioral tests\n N={len(c_b_df_high_p_tau)} - High p-tau', fontsize=18)
plt.savefig(f'{SAVE_DIR}/Cognitive_and_Behavioral_pearson_correlation_heatmap__high_p_tau_no_annotation_stars_only.pdf', bbox_inches='tight', dpi=300, format='pdf')

# Basline - If we would screen the population based on MOCA or MBI what will be the detection rate for high p-tau individuals?

In [ ]:
# MOCA
high_tau_mci = df[(df[TAU_217] > TAU_217_POPULATION_THR) & (df[MOCA_COL] <= MCI_THR)]
low_tau_mci = df[(df[TAU_217] < TAU_217_POPULATION_THR ) & (df[MOCA_COL] <= MCI_THR)]

high_tau_nci = df[(df[TAU_217] > TAU_217_POPULATION_THR) & (df[MOCA_COL] > MCI_THR)]
low_tau_nci = df[(df[TAU_217] < TAU_217_POPULATION_THR) & (df[MOCA_COL] > MCI_THR)]

# MBI
high_tau_mbi = df[(df[TAU_217] > TAU_217_POPULATION_THR) & (df[MBI_C_COL] > MBI_C_THR)]
low_tau_mbi = df[(df[TAU_217] < TAU_217_POPULATION_THR) & (df[MBI_C_COL] > MBI_C_THR)]


high_tau_non_mbi = df[(df[TAU_217] > TAU_217_POPULATION_THR) & (df[MBI_C_COL] <= MBI_C_THR)]
low_tau_non_mbi = df[(df[TAU_217] < TAU_217_POPULATION_THR) & (df[MBI_C_COL] <= MBI_C_THR)]


In [ ]:
low_tau_mci.shape, high_tau_mci.shape

In [ ]:
def plot_baseline_normalized(tn, fp, fn, tp, title='Baseline Confusion Matrix (Normalized by p-tau)', 
                            x_tics_labels=['High MoCa', 'Low MoCa'], y_tics_labels=['Low p-tau', 'High p-tau']):
    """
    Plots a confusion matrix-style heatmap, normalized by the sum of p-tau group (row) counts.
    tn, fp = Low p-tau (row)
    fn, tp = High p-tau (row)
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Calculate counts
    row1_total = len(tn) + len(fp)
    row2_total = len(fn) + len(tp)
    # Avoid division by zero
    row1_total = row1_total if row1_total != 0 else 1
    row2_total = row2_total if row2_total != 0 else 1

    # Normalize each row by the total number in that p-tau group (row)
    matrix = np.array([
        [len(tn) / row1_total, len(fp) / row1_total],
        [len(fn) / row2_total, len(tp) / row2_total]
    ])
    
    # Display as % for better interpretability in a heatmap
    annot_matrix = np.char.add(
        (100 * matrix).round(1).astype(str), "%"
    )

    sns.heatmap(matrix, annot=annot_matrix, fmt='', cmap='Blues',
                xticklabels=x_tics_labels, 
                yticklabels=y_tics_labels, 
                ax=ax, vmin=0, vmax=1, rasterized=False)

    ax.set_title(title)
    ax.set_xlabel(MOCA_COL)
    ax.set_ylabel(TAU_217)
    plt.savefig(SAVE_DIR + 'Baseline_Confusion_Matrix_Normalized_by_ptau.pdf', bbox_inches='tight', dpi=300, format='pdf')

plot_baseline_normalized(low_tau_nci, low_tau_mci, high_tau_nci, high_tau_mci)

# plot_baseline_normalized(high_tau_mbi, low_tau_mbi, high_tau_non_mbi, low_tau_non_mbi, x_tics_labels=['Low MBI-C', 'High MBI-C'])

In [ ]:
MCI_THR

In [ ]:
mci_and_mbi = c_b_df_high_p_tau[(c_b_df_high_p_tau["Negative MOCA_score"] <= -MCI_THR) & (c_b_df_high_p_tau[MBI_C_COL] == 1)]
healthy = c_b_df_high_p_tau[(c_b_df_high_p_tau["Negative MOCA_score"] > -MCI_THR) & (c_b_df_high_p_tau[MBI_C_COL] == 0)]
only_mci = c_b_df_high_p_tau[(c_b_df_high_p_tau["Negative MOCA_score"] <= -MCI_THR) & (c_b_df_high_p_tau[MBI_C_COL] == 0)]
onl_mbi = c_b_df_high_p_tau[(c_b_df_high_p_tau["Negative MOCA_score"] > -MCI_THR) & (c_b_df_high_p_tau[MBI_C_COL] == 1)]

# Make a bar plot of the number of people in each group
plt.figure(figsize=(8, 6))
plt.bar(['MCI and MBI', 'Healthy', 'Only MCI', 'Only MBI'], [len(mci_and_mbi), len(healthy), len(only_mci), len(onl_mbi)])
plt.title('Number of people in each group')
plt.show()

In [ ]:
from turtle import color

from seaborn import palettes


def plot_mci_vs_mbi(df, title='MBI vs MCI'):
    mci_and_mbi = df[(df["Negative MOCA_score"] > -MCI_THR) & (df[MBI_C_COL] == 1)]
    healthy = df[(df["Negative MOCA_score"] <= -MCI_THR) & (df[MBI_C_COL] == 0)]
    only_mci = df[(df["Negative MOCA_score"] <= -MCI_THR) & (df[MBI_C_COL] == 1)]
    onl_mbi = df[(df["Negative MOCA_score"] > -MCI_THR) & (df[MBI_C_COL] == 0)]

    # Create a confusion matrix-style plot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Create a 2x2 matrix
    matrix = np.array([[len(onl_mbi), len(mci_and_mbi)], 
                    [len(healthy), len(only_mci)]])

    # Plot the confusion matrix
    sns.heatmap(matrix, annot=True, fmt='g', cmap='Blues',
                xticklabels=['Cognitive Healthy', 'MCI'], 
                yticklabels=['MBI', 'Behavioral Healthy'], 
                ax=ax, rasterized=False)

    ax.set_title(title)
    ax.set_xlabel(COGNITIVE_COLUMN)
    ax.set_ylabel(BEHAVIORAL_COLUMN)
    plt.savefig(SAVE_DIR + 'Cognitive_and_Behavioral_pearson_correlation_heatmap_mci_vs_mbi.pdf', bbox_inches='tight', dpi=300, format='pdf')

plot_mci_vs_mbi(c_b_df)

In [ ]:
df_high_vs_low_217 = df_217[(df_217['tau_217_categorized'] == 2) | (df_217['tau_217_categorized'] == 0)]

plot_mci_vs_mbi(df_high_vs_low_217, title='MBI vs MCI for high vs low p-tau217')

In [ ]:
# remove rows with nan values in iADL from df_217
df_217_no_nan = df_217.dropna(subset=[BEHAVIORAL_COLUMN, COGNITIVE_COLUMN])